# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begs44/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My rule in plain words:** A page that Google already shows a lot (≥ 500 impressions, position 1–20) but that gets far fewer clicks than other pages at the same position is the first one to fix. The action is to rewrite its title and meta description.

**Reason code:** `low_ctr_for_position` (pages that are not flagged get `no_flag` → `monitor`).

**Signal checks (tables below, n printed; the decline label is used only to check, never as an input):**

- **Signal 1: Staleness (`days_since_last_update`). This is the signal behind FlyRank's refresh flag (≥ 180 days). Verdict: MIXED.** The 90–119 day bucket declines more than fresh pages (0.613 vs 0.511, n = 9,115), and this holds inside 12 of 17 clients. But the real flag threshold of 180+ days gives 0.471, which is *below* the 0.542 base rate (n = 174). Also, the column is basically two values (20 and 104 days), which looks like batch updates, not real ageing. So I did **not** put staleness in my rule.
- **Signal 2: CTR relative to position. This is the signal behind FlyRank's CTR-fix logic. Verdict: CONFIRMED.** Among visible pages, the quintile with the lowest CTR for its rank declines at 0.738, and the highest quintile at 0.492. The pattern goes down step by step, with n ≈ 2,400 per quintile. It holds inside 14 of 17 clients. This is the signal my rule is built on.

In [2]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/begs44/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Evaluation label ONLY — used to check signals and score the queue afterwards, NEVER as a rule input.
df["target_decay"] = (df["trend_direction"] == "down").astype(int)
base = df["target_decay"].mean()
print(f"Rows: {len(df):,} | base decline rate: {base:.3f}\n")

# ---------- Signal 1: STALENESS (behind FlyRank's refresh flag: >=180 days since update) ----------
stale_bins   = [0, 30, 90, 120, 180, 10_000]
stale_labels = ["0-29d", "30-89d", "90-119d", "120-179d", "180d+ (flag)"]
df["stale_bucket"] = pd.cut(df["days_since_last_update"], stale_bins, labels=stale_labels, right=False)

t1 = df.groupby("stale_bucket", observed=True)["target_decay"].agg(n="size", decline_rate="mean")
t1["vs_base"] = t1["decline_rate"] - base
print("SIGNAL 1 — staleness (days_since_last_update)")
print(t1.round(3).to_string(), "\n")

# Same check inside each client (is it a client effect?)
df["updated_90_119d"] = df["days_since_last_update"].between(90, 119)
by_client = df.groupby(["client_id", "updated_90_119d"])["target_decay"].mean().unstack().dropna()
print(f"Within-client: 90-119d bucket declines more than 0-29d in "
      f"{(by_client[True] > by_client[False]).sum()} of {len(by_client)} clients that have both")
print("Top update values:", df["days_since_last_update"].value_counts().head(3).to_dict(), "\n")

# ---------- Signal 2: CTR vs POSITION (behind FlyRank's CTR-fix flag: visible page, low CTR for its rank) ----------
visible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)  # avg_position 0 = no data
df["pos_band"] = pd.cut(df["avg_position"], [0, 3, 5, 10, 15, 20])
expected_ctr = df[visible].groupby("pos_band", observed=True)["ctr"].median()   # label-free benchmark
print("Median CTR (%) per position band, visible pages:", {str(k): round(x, 2) for k, x in expected_ctr.items()})

df["ctr_vs_expected"] = df["ctr"] / df["pos_band"].map(expected_ctr).astype(float)
v = df[visible].copy()
v["ctr_quintile"] = pd.qcut(v["ctr_vs_expected"].rank(method="first"), 5,
                            labels=["Q1 lowest CTR for rank", "Q2", "Q3", "Q4", "Q5 highest CTR for rank"])
t2 = v.groupby("ctr_quintile", observed=True)["target_decay"].agg(n="size", decline_rate="mean")
t2["vs_base"] = t2["decline_rate"] - base
print("\nSIGNAL 2 — CTR relative to position (visible pages: >=500 impressions, position 1-20)")
print(t2.round(3).to_string(), "\n")

v["low_ctr"] = v["ctr_vs_expected"] < 1
cc = v.groupby(["client_id", "low_ctr"])["target_decay"].agg(["mean", "size"]).unstack()
cc = cc[(cc["size"][True] >= 20) & (cc["size"][False] >= 20)]
print(f"Within-client: below-expected CTR declines more in "
      f"{(cc['mean'][True] > cc['mean'][False]).sum()} of {len(cc)} clients (>=20 pages each side)\n")

print("VERDICT signal 1 (staleness):    MIXED")
print("VERDICT signal 2 (CTR vs rank):  CONFIRMED")

Rows: 30,000 | base decline rate: 0.542

SIGNAL 1 — staleness (days_since_last_update)
                  n  decline_rate  vs_base
stale_bucket                              
0-29d         20480         0.511   -0.031
30-89d          175         0.589    0.047
90-119d        9115         0.613    0.071
120-179d         56         0.286   -0.256
180d+ (flag)    174         0.471   -0.071 

Within-client: 90-119d bucket declines more than 0-29d in 12 of 17 clients that have both
Top update values: {20: 11573, 104: 8773, 22: 3564} 

Median CTR (%) per position band, visible pages: {'(0, 3]': 0.2, '(3, 5]': 0.33, '(5, 10]': 0.21, '(10, 15]': 0.18, '(15, 20]': 0.16}

SIGNAL 2 — CTR relative to position (visible pages: >=500 impressions, position 1-20)
                            n  decline_rate  vs_base
ctr_quintile                                        
Q1 lowest CTR for rank   2405         0.738    0.196
Q2                       2404         0.628    0.086
Q3                       2405    

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Score** = CTR shortfall × log10(impressions), only for eligible pages (≥ 500 impressions, position 1–20).
CTR shortfall = 1 − (page CTR ÷ median CTR of its position band), clipped to 0–1. No fitted weights.
Inputs: `impressions_90d`, `avg_position`, `ctr`. Nothing else. The queue is written to `work/outputs/baseline_action_score.csv`.

**Honest check:** precision@50 = 0.64 and precision@100 = 0.71, against a base rate of 0.542 (0.599 among eligible pages). precision@10 is only 0.50. Section 4 explains why. This is the number my Week-5 model has to beat.

In [3]:
import os, json

# ---------- ONE rule: low CTR for its position, on a page people actually see ----------
RULE_INPUTS = ["impressions_90d", "avg_position", "ctr"]          # nothing else enters the score

eligible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
shortfall = (1 - df["ctr_vs_expected"]).clip(lower=0, upper=1)      # 0 = at/above expected CTR, 1 = zero clicks
df["action_score"] = np.where(eligible, shortfall * np.log10(df["impressions_90d"]), 0.0).round(4)

flagged = df["action_score"] > 0
df["reason_code"]  = np.where(flagged, "low_ctr_for_position", "no_flag")
df["action_label"] = np.where(flagged, "rewrite_title_and_meta", "monitor")

queue = df.sort_values(["action_score", "impressions_90d"], ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

out_cols = ["rank", "content_id", "client_id", "action_score", "reason_code", "action_label",
            "impressions_90d", "clicks_90d", "avg_position", "ctr", "ctr_vs_expected",
            "days_since_last_update", "content_type", "main_intent"]
os.makedirs("work/outputs", exist_ok=True)
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote work/outputs/baseline_action_score.csv  ({len(queue):,} rows, {flagged.sum():,} flagged)\n")

# ---------- Honest check: precision@K vs base rate (label used ONLY here) ----------
metrics = {"rows": int(len(queue)), "flagged": int(flagged.sum()),
           "base_rate_all": round(float(base), 3),
           "base_rate_eligible": round(float(df.loc[eligible, "target_decay"].mean()), 3)}
for k in [10, 50, 100, 500]:
    metrics[f"precision_at_{k}"] = round(float(queue["target_decay"].head(k).mean()), 3)
for key, val in metrics.items():
    print(f"{key:>20}: {val}")

with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump({"rule": "low_ctr_for_position", "inputs": RULE_INPUTS, **metrics}, f, indent=2)

Wrote work/outputs/baseline_action_score.csv  (30,000 rows, 5,831 flagged)

                rows: 30000
             flagged: 5831
       base_rate_all: 0.542
  base_rate_eligible: 0.599
     precision_at_10: 0.5
     precision_at_50: 0.64
    precision_at_100: 0.71
    precision_at_500: 0.714


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Every row: action = `rewrite_title_and_meta`, reason = `low_ctr_for_position`.

1. **208,678 impressions, 0 clicks, pos 9.7.** Why: the biggest page with zero clicks. Wrong if: clicks are being credited to another URL (canonical or redirect). Then it is a tracking fix, not a title fix.
2. **140,079 impr, 16 clicks, pos 7.6.** Why: huge visibility, almost no clicks. Wrong if: the impressions come from loosely related long-tail queries that a new title will not win.
3. **128,068 impr, 17 clicks, pos 2.2.** Why: position 2 with 0.01% CTR. Wrong if: the average position hides the query mix (high only on a few odd queries). It was not declining. **Weak pick.**
4. **112,434 impr, 10 clicks, pos 7.2.** Why: same pattern as #2. Wrong if: the SERP is full of features (AI overview, video) that take the clicks no matter what the title says.
5. **272,144 impr, 75 clicks, pos 2.3.** Why: the largest page in the queue, with a low CTR. Wrong if: a featured snippet or AI answer satisfies the query in Google. It was not declining. **Weak pick.**
6. **119,217 impr, 26 clicks, pos 7.0, updated 7 days ago.** Why: low CTR for its rank. Wrong if: the last update has not been re-indexed yet. Wait before rewriting again.
7. **223,271 impr, 70 clicks, pos 7.8.** Why: very visible, low CTR. Wrong if: the query is informational and answered on the results page itself.
8. **65,138 impr, 6 clicks, pos 6.8.** Why: almost zero CTR on page 1. Wrong if: impressions come from image search or another surface where CTR is naturally tiny.
9. **46,879 impr, 7 clicks, pos 3.7.** Why: top-5 position, nearly no clicks. Wrong if: it is the same position/query-mix illusion as #3.
10. **56,363 impr, 3 clicks, pos 5.9, transactional.** Why: the only transactional page in the top 10. Wrong if: ads above the organic results take the clicks. A title change will not fix that.

Hindsight (label, not an input): 5 of 10 were declining. That is no better than the base rate at the very top.

In [4]:
top10 = queue.head(10)[["rank", "action_label", "reason_code", "action_score", "impressions_90d",
                        "clicks_90d", "avg_position", "ctr", "days_since_last_update",
                        "content_type", "main_intent", "target_decay"]]
print(top10.to_string(index=False))
print(f"\nHindsight (label, not an input): {top10['target_decay'].sum()} of 10 were declining")

 rank           action_label          reason_code  action_score  impressions_90d  clicks_90d  avg_position  ctr  days_since_last_update    content_type   main_intent  target_decay
    1 rewrite_title_and_meta low_ctr_for_position        5.3195           208678           0           9.7 0.00                     104 keyword article informational             1
    2 rewrite_title_and_meta low_ctr_for_position        4.9013           140079          16           7.6 0.01                      20 keyword article informational             1
    3 rewrite_title_and_meta low_ctr_for_position        4.8521           128068          17           2.2 0.01                     104 keyword article informational             0
    4 rewrite_title_and_meta low_ctr_for_position        4.8104           112434          10           7.2 0.01                      20 keyword article informational             1
    5 rewrite_title_and_meta low_ctr_for_position        4.6196           272144          75        

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:**
- **#3 and #5 sit at position ~2 with 0.01–0.03% CTR.** That is not believable for a real page-1 listing. A 90-day average position hides which queries the page ranks for. Neither page was declining.
- **The log10(impressions) weight puts the biggest pages first,** but pages with 3,000+ impressions decline less than 500–3,000 ones (0.57 vs 0.62). That is why precision@10 (0.50) is weaker than precision@100 (0.71).
- **Zero-click flagged pages decline more** (0.773, n = 1,215) than flagged pages with some clicks (0.637, n = 4,616). The rule is right about them, but some of these may be tracking problems, not title problems.
- **One client holds 17 of the top 50.** The queue partly reflects one site, not a general pattern.
- **#6 was updated 7 days ago.** A rule that also checked recency would hold it back.

**Leakage check:** the score uses only `impressions_90d`, `avg_position` and `ctr`. No `trend_direction`, no `trend_pct`, no 30-day trend-window column, and no IDs. The code below asserts this. One honest caveat: the 90-day totals overlap the 30-day window the label is built from. These are the course's standard features, but they are not strictly "before" the label, so my Week-5 model has to live under the same constraint. All claims are observed and directional, for decision support only, not causal.

In [5]:
# ---------- Weak picks ----------
top50 = queue.head(50)
print("Top-50 pages per client (concentration):")
print(top50["client_id"].value_counts().head(5).to_string(), "\n")

zero_click = queue[queue["action_score"] > 0]
zc = zero_click.assign(zero_clicks=zero_click["clicks_90d"] == 0)
print("Flagged pages — zero clicks vs some clicks:")
print(zc.groupby("zero_clicks")["target_decay"].agg(n="size", decline_rate="mean").round(3).to_string(), "\n")

# ---------- Leakage check ----------
FORBIDDEN = {"trend_direction", "trend_pct", "target_decay", "is_declining_label",
             "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
             "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
             "content_id", "client_id"}
leaks = FORBIDDEN & set(RULE_INPUTS)
assert not leaks, f"LEAK: {leaks}"
print("Rule inputs:", RULE_INPUTS)
print("Leakage check passed — no label, label-source, 30-day trend-window or ID column in the score.")

Top-50 pages per client (concentration):
client_id
client_19581e27de    17
client_f369cb89fc     8
client_4e07408562     7
client_7f2253d7e2     4
client_6208ef0f77     3 

Flagged pages — zero clicks vs some clicks:
                n  decline_rate
zero_clicks                    
False        4616         0.637
True         1215         0.773 

Rule inputs: ['impressions_90d', 'avg_position', 'ctr']
Leakage check passed — no label, label-source, 30-day trend-window or ID column in the score.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.